<a href="https://colab.research.google.com/github/pandahsu849/Programing_Languages/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [18]:
!pip install -U google-generativeai

In [1]:
!pip install -q google-generativeai

In [9]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [10]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [11]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing"
WORKSHEET_NAME = "工作表3"
REQUIRED_COLUMNS = ["日期", "科目", "成績", "AI 摘要"] # Modified to include "AI 摘要"

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [21]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash-lite'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to perform tasks and make decisions.


### 定義 AI 摘要函式

In [22]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要、簡評與建議。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [23]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # Check if the first row is empty. If so, write headers.
    try:
        if not _ws.row_values(1): # Check if the first row has any values (i.e., headers)
            _ws.append_row(REQUIRED_COLUMNS) # Add headers if the first row is empty
            print(f"--- Google Sheet: Adding headers: {REQUIRED_COLUMNS} ---")
    except Exception as e:
        print(f"--- Google Sheet: Error checking or adding headers: {e} ---")
        # Continue even if adding headers fails, as grade append might still work.

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    initial_row_count = len(_ws.col_values(1)) # Get current number of rows before appending

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        return sheet_message, "" # If grades fail to write, return here


    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    try:
        # 找到剛剛寫入的最後一筆成績的列數
        # Assuming new_grades_for_sheet is not empty
        if new_grades_for_sheet:
            last_grade_row_index = initial_row_count + len(new_grades_for_sheet)
            # 將 AI 摘要寫入該列的第四欄
            _ws.update_cell(last_grade_row_index, 4, summary) # Column 4 is "AI 摘要"
            sheet_message += "AI 摘要已成功寫入 Google Sheet。"
        else:
            sheet_message += "沒有成績可供寫入 AI 摘要。"

    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [24]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的學生成績所產生的簡單摘要、簡評與建議：

**成績摘要：**

*   **國文：** 85 分
*   **數學：** 78 分
*   **英文：** 92 分

**簡評：**

整體而言，這位學生在英文科目上表現非常突出，取得了優異的成績。國文成績也相當不錯，屬於良好的水準。在數學科目上，表現則相對較為平穩。

**建議：**

*   **維持英文優勢：** 鼓勵學生繼續保持對英文的學習熱情，深入探索更多元的學習資源，例如閱讀英文原著、觀看無字幕影集或參與英文交流活動，以期能更上一層樓。
*   **鞏固國文基礎：** 繼續保持國文學習的良好習慣，可以嘗試擴展閱讀的類型，或針對較弱的文法、寫作技巧加強練習，以確保穩定的學術表現。
*   **加強數學學習：** 建議針對數學科目，回顧並釐清觀念，尤其是在較弱的部分，可以尋求額外的輔導或練習，增加解題的熟練度，以期能提升分數。

Google Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。


定義 Gradio 處理函式

In [29]:
import gradio as gr

def add_to_list(subject, score, current_data):
    if not subject or not score:
        return current_data, subject, score  # 如果沒填完整就不動作

    # 將新資料加入狀態列表
    new_data = current_data + [[subject, score]]
    # 清空輸入框，方便繼續打下一筆
    return new_data, "", ""

# 建立一個函數來處理送出後清空清單的動作
def submit_and_clear(data):
    # 呼叫你原本的處理函數
    sheet_msg, ai_summary = process_grades_and_summary(data)
    # 送出後，清空暫存清單 (回傳空列表給 State 和 Dataframe)
    return sheet_msg, ai_summary, []

with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在左側輸入科目和成績，點擊『新增至清單』。確認下方清單無誤後，再點擊『送出整批資料』。")

    # 這個 State 用來在背景記住目前累積了哪些資料
    data_state = gr.State([])

    with gr.Row():
        # 左側：輸入區
        with gr.Column(scale=1):
            gr.Markdown("### 1. 輸入區 ")
            with gr.Row():
                sub_input = gr.Textbox(label="科目", placeholder="例如：國文")
                score_input = gr.Textbox(label="成績", placeholder="例如：90")

            add_btn = gr.Button("⬇️ 新增至清單", variant="secondary")

            gr.Markdown("### 2. 待送出清單")
            grade_table = gr.Dataframe(
                headers=["科目", "成績"],
                value=[],
                datatype=["str", "str"],
                interactive=False, # 這裡純顯示，避免點擊干擾
                label="目前已輸入的成績"
            )

            submit_button = gr.Button("🚀 送出整批資料", variant="primary")

        # 右側：結果輸出區
        with gr.Column(scale=1):
            gr.Markdown("### 3. 執行結果")
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    # 事件綁定：新增資料到清單
    add_btn.click(
        fn=add_to_list,
        inputs=[sub_input, score_input, data_state],
        outputs=[data_state, sub_input, score_input] # 更新 State 並清空輸入框
    ).then( # 使用 .then() 確保 State 更新後才刷新表格顯示
        fn=lambda x: x, # 簡單的回傳函數，把 State 的值丟給表格
        inputs=data_state,
        outputs=grade_table
    )

    # 事件綁定：送出資料到 Google Sheet 和 AI
    submit_button.click(
        fn=submit_and_clear,
        inputs=data_state, # 把累積的資料送出去
        outputs=[sheet_output, summary_output, data_state] # 接收訊息並清空 State
    ).then(
        fn=lambda x: x, # 同樣確保清空後的 State 反映到表格上
        inputs=data_state,
        outputs=grade_table
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c86e9e33fd0f41b966.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- 正在呼叫 AI 模型生成摘要... ---
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c86e9e33fd0f41b966.gradio.live
